# VectorDB, Hugging Face & Ollama | Assignment

**Question 1: What is a Vector Database (VectorDB) and how is it different from** **traditional** **databases?**

A Vector Database (VectorDB) is a specialized database designed to store and search high-dimensional vectors (numerical representations of data like text, images, or audio embeddings generated by AI models). Unlike traditional databases, which store structured data in rows and columns and rely on exact matches (e.g., SQL queries), vector databases focus on similarity search—finding items that are “closest” in meaning or context using distance metrics like cosine similarity. This makes VectorDBs ideal for applications like semantic search, recommendation systems, and AI-powered retrieval, whereas traditional databases are better suited for structured data storage and precise queries.


**Question 2: Explain the various types of VectorDBs available and describe their** **suitability for different use cases**

There are several types of Vector Databases, mainly differing by architecture and use case. Standalone vector databases like Pinecone or Weaviate are purpose-built for fast similarity search and are ideal for applications like semantic search, recommendation systems, and LLM retrieval (RAG). Vector-enabled traditional databases such as PostgreSQL (with pgvector) or MongoDB Atlas combine structured data with vector search, making them suitable when you need both relational queries and embeddings in one system. Search engine-based vector DBs like Elasticsearch and OpenSearch are best for hybrid search (keyword + semantic), commonly used in enterprise search and e-commerce. Lastly, in-memory/vector libraries such as FAISS or Annoy are lightweight and efficient for experimentation or small-scale systems but lack full database features like persistence and scalability.

**Question 3: Why is Chroma DB considered important in the context of AI/ML projects? Describe its key features**.

Chroma (Chroma DB) is important in AI/ML projects because it is specifically designed to store and retrieve embeddings efficiently, which are core to applications like LLMs, semantic search, and Retrieval-Augmented Generation (RAG). It simplifies the process of building AI systems by providing an easy way to manage vector data without complex setup.

Key features include embedding storage and similarity search (to find contextually similar data), tight integration with LLM frameworks (like LangChain), lightweight and developer-friendly setup (runs locally without heavy infrastructure), metadata filtering (combine vector search with structured filters), and persistence support (store data for reuse). This makes Chroma DB especially suitable for rapid prototyping, chatbots, document search, and AI-powered applications.


**Question 4: What are the benefits of using Hugging Face Hub for generative AI tasks?**

Hugging Face Hub provides several benefits for generative AI tasks by acting as a centralized platform to access, share, and deploy models. It offers a large collection of pre-trained models (for text, image, audio generation), which saves time and compute compared to training from scratch. It supports easy integration with libraries like Transformers, enabling quick experimentation and deployment. The Hub also provides version control and collaboration features, making it easier for teams to manage models and datasets. Additionally, it includes inference APIs and hosted spaces, allowing developers to test and deploy generative AI applications without heavy infrastructure, making it highly suitable for rapid development and scaling.

**Question 5: Describe the process and advantages of navigating and using pre-trained models from the Hugging Face Hub**.

Using pre-trained models from the Hugging Face Hub involves a simple process. First, you search and explore models based on task (e.g., text generation, summarization, image generation) and filters like language, size, or popularity. Next, you review model details such as documentation, usage examples, and evaluation metrics. Then, you can load the model بسهولة using libraries like Transformers (e.g., with a few lines of Python code) or via API for direct inference. Finally, you can fine-tune or customize the model on your own dataset if needed and deploy it in applications.

The advantages include saving time and resources (no need to train from scratch), access to state-of-the-art models, ease of use with ready-to-run code, and strong community support with shared improvements and updates. It also enables rapid prototyping and deployment, making it ideal for building generative AI applications efficiently.


In [ ]:
# Question 6: Install and set up Chroma DB, and insert sample vector data for semantic search.
!pip install chromadb

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Create a Chroma client
client = chromadb.Client()

# Use default embedding model
embedding_function = embedding_functions.DefaultEmbeddingFunction()

# Create a collection
collection = client.create_collection(
    name="my_collection",
    embedding_function=embedding_function
)

In [ ]:
documents = [
    "I love machine learning",
    "Artificial intelligence is the future",
    "Data science is amazing"
]

ids = ["1", "2", "3"]

collection.add(
    documents=documents,
    ids=ids
)

In [ ]:
query = "AI and ML"

results = collection.query(
    query_texts=[query],
    n_results=2
)

print(results)

In [ ]:
# Question 7: Demonstrate how to download and fine-tune a Hugging Face model for a text generation task.

!pip install transformers datasets torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"   # free & open-source

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ✅ FIX: set padding token
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length")

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Data collator (important for causal LM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # because it's a causal language model
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_dir="./logs",
    save_steps=500,
    save_total_limit=2
)

small_train_dataset = tokenized_datasets["train"].select(range(200))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,  # ✅ only 500 samples
    data_collator=data_collator
)

# ✅ Train
trainer.train()

In [ ]:
trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./fine_tuned_model",
    tokenizer=tokenizer
)

result = generator(
    "Artificial Intelligence is",
    max_length=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])

In [ ]:
# Question 8: Create a custom LLM using Ollama and Llama2, and run it locally for basic text prompts.

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess

subprocess.Popen("ollama serve", shell=True)

!ollama pull llama2

!ollama run llama2 "Explain AI in simple terms"

modelfile = """
FROM llama2
SYSTEM "You are a helpful AI tutor who explains concepts simply."
PARAMETER temperature 0.7
"""

with open("Modelfile", "w") as f:
    f.write(modelfile)

!ollama create my-llm -f Modelfile
!ollama run my-llm "What is machine learning?"

In [ ]:
# Question 9: Implement a basic RAG (Retrieval-Augmented Generation) system using Ollama with Llama3.
!pip install chromadb langchain ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
subprocess.Popen("ollama serve", shell=True)

!ollama pull llama3

documents = [
    "Machine learning is a subset of AI.",
    "Deep learning uses neural networks.",
    "RAG combines retrieval with generation.",
    "LLMs are used for text generation tasks."
]

import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

embedding_function = embedding_functions.DefaultEmbeddingFunction()

collection = client.create_collection(
    name="rag_collection",
    embedding_function=embedding_function
)

collection.add(
    documents=documents,
    ids=[str(i) for i in range(len(documents))]
)

def retrieve(query):
    results = collection.query(
        query_texts=[query],
        n_results=2
    )
    return results["documents"][0]

import subprocess

def generate_answer(query):
    context = retrieve(query)

    prompt = f"""
    Answer the question based on the context below:

    Context: {context}

    Question: {query}
    """

    result = subprocess.run(
        ["ollama", "run", "llama3", prompt],
        capture_output=True,
        text=True
    )

    return result.stdout
query = "What is RAG?"

answer = generate_answer(query)
print(answer)

**Question 10: A health-tech startup wants to build a chatbot that can answer user queries based on medical research articles. Propose and explain a solution using Hugging Face models for understanding, VectorDB for retrieval, and Ollama for generation**

A suitable solution is to build a RAG-based chatbot using Hugging Face Hub, Chroma, and Ollama with Llama 3. First, medical research articles are collected, cleaned, and split into smaller chunks for processing. These chunks are then converted into embeddings using Hugging Face models (like sentence-transformers), which capture the semantic meaning of the text.

The embeddings are stored in a vector database such as Chroma, enabling efficient similarity search. When a user asks a question, it is converted into an embedding and matched against stored vectors to retrieve the most relevant medical content. This ensures that the system fetches contextually accurate information instead of relying only on keywords.

Finally, the retrieved context along with the user query is passed to Llama 3 via Ollama to generate a response. The model is prompted to answer strictly based on the provided medical context and include safety instructions (e.g., consult a doctor if unsure). This approach improves accuracy, maintains privacy (local inference), and provides reliable, context-aware answers for healthcare use cases.